In [ ]:
from utils import *

In [ ]:
model = "deepseek-ai/deepseek-coder-1.3b-base"
short_model_name = model.split("/")[-1]
model_results_dir = f"../model_results/{short_model_name}"
# Code snippets directories
code_snippets_dir = "../code_snippets"
code_snippets_with_data_driven_aois_dir = os.path.join(model_results_dir, "code_snippets_with_data_driven_aois")

# EEG data directory and parameters
data_driven_processed_eeg_dir = "" # Adjust this path to the new processed data directory after setting data driven frps
t_min, t_max = 0.4, 0.8
channel = 'Fz'
downsample_freq = None

# Output directories for results
analysis_results_dir = os.path.join(model_results_dir, "individual_analysis")

if analysis_results_dir and not os.path.exists(analysis_results_dir):
    os.makedirs(analysis_results_dir)


In [ ]:
compute_perplexity_results = False
annotate_with_aois = True
load_eeg_data = True

In [ ]:
if compute_perplexity_results:
    with open("../hf_token", "r") as f:
        hf_token = f.readline().strip()
    # login to huggingface
    if not hf_token:
        raise ValueError("Hugging Face token is not set. Please provide a valid token in the hf_token file.")
    print("Logging in to Hugging Face...")
    os.environ["HF_TOKEN"] = hf_token
    os.system(f"echo {hf_token} | huggingface-cli login --token {hf_token}")
    print("Logged in to Hugging Face successfully.")
    print("Loading Java files for perplexity computation...")
    code_without_aois = read_java_files(code_snippets_dir)
    print(f"Loaded {len(code_without_aois)} Java files for perplexity computation.")
    results = process_files(
        code_without_aois,
        model_name=model,
        save_path=f"{model_results_dir}/perplexity_results.json",
        limit=None
    )

if annotate_with_aois:
    # Annotate the perplexity results with AOI markers
    code_with_aois = read_java_files(code_snippets_with_data_driven_aois_dir)
    results = map_aois_to_perplexity_file(
        input_path=f"{model_results_dir}/perplexity_results_with_aois.json",
        output_path=f"{model_results_dir}/perplexity_results_with_data_driven_aois.json",
        file_dict_with_aoi_markers=code_with_aois,
        aoi_types=['1', '2', '3', '4', '5', '6', '7', '8', '9'],
    )

In [ ]:
if load_eeg_data:
    from eeg_loader import EEGDataLoader
    # Create the EEGDataLoader instance
    eeg_loader = EEGDataLoader(data_driven_processed_eeg_dir)
    X, y, groups, trial_ids = eeg_loader.load_all_participants_individual_snippets(
        t_min=t_min, t_max=t_max, channel=channel, downsample_freq=downsample_freq)

    # Get both mean and max EEG signals
    averaged_trials, std_trials = eeg_loader.average_trials(X=X, trial_ids=trial_ids)
    max_trials = eeg_loader.max_trials(X=X, trial_ids=trial_ids)
else:
    # Load precomputed trials
    averaged_trials = None
    std_trials = None
    max_trials = None

In [ ]:
with open(f"{model_results_dir}/perplexity_results_with_data_driven_aois.json", "r") as f:
    results = json.load(f)

analyze_individual_snippets(results=results, java_files_dir_with_aois=code_snippets_with_data_driven_aois_dir,
                            output_dir=analysis_results_dir, averaged_trials=averaged_trials,
                            std_trials=std_trials, t_min=t_min, t_max=t_max)

In [ ]:
analysis_df = create_eeg_perplexity_correlation_analysis_df_individual_snippets(
    results_path=f"{analysis_results_dir}/perplexity_stats.csv",
    output_dir=analysis_results_dir,
    averaged_trials=averaged_trials
)